# Online Shoppers Purchasing Intention — ML End-term
## 2.2 Data Analysis and Dimensionality Reduction

**Methods:** PCA and LDA on the mixed representation; TruncatedSVD on the full binary representation. This notebook builds exploratory plots from the complete cleaned dataset. **Do not feed the exploratory fitted transforms into the held-out classification evaluations in §2.4.** For evaluation, preprocessing and reduction must be fitted only on each training split.

Put this notebook, `preprocess_data.py`, and `online_shoppers_intention.csv` in the same folder. Run cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import roc_auc_score
from preprocess_data import (explore_data, get_mixed_pipeline,
                             get_full_binary_pipeline, get_full_discrete_pipeline)

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TARGET = 'Revenue'

### 2.2.1 Data overview and basic statistics

In [ ]:
data = explore_data(pd.read_csv('online_shoppers_intention.csv'), verbose=True)
X = data.drop(columns=[TARGET])
y = data[TARGET]
print('Original input features:', X.shape[1])
print('Target proportions (%):')
display((y.value_counts(normalize=True) * 100).round(2).rename('percent'))

In [ ]:
mixed_pipe = get_mixed_pipeline()
X_mixed = mixed_pipe.fit_transform(X)  # Exploratory fit ONLY
mixed_features = mixed_pipe.get_feature_names_out()
X_mixed_df = pd.DataFrame(X_mixed, columns=mixed_features, index=X.index)

binary_pipe = get_full_binary_pipeline()
X_binary = binary_pipe.fit_transform(X)  # Exploratory fit ONLY
binary_features = binary_pipe.get_feature_names_out()

print('Mixed shape:', X_mixed.shape)
print('Binary shape:', X_binary.shape)
print('Binary is sparse:', sparse.issparse(X_binary))
if sparse.issparse(X_binary):
    print('Binary density:', X_binary.nnz / (X_binary.shape[0] * X_binary.shape[1]))
print('No NaNs mixed:', not X_mixed_df.isna().any().any())
display(X_mixed_df.describe().T.head(20))

The standardized numeric features have mean approximately zero and unit population standard deviation, while one-hot features remain 0/1. Discrete encodings can produce many correlated indicator columns. Statistics below describe the **transformed input space**, not the target.

### 2.2.2 PCA and LDA on mixed features

In [ ]:
PCA_6 = PCA(n_components=6, random_state=RANDOM_STATE)
X_PCA_6 = PCA_6.fit_transform(X_mixed)
LDA_1 = LDA(n_components=1)
X_LDA_1 = LDA_1.fit_transform(X_mixed, y)

print('PCA reduced shape:', X_PCA_6.shape)
print('LDA reduced shape:', X_LDA_1.shape)
print('PCA explained variance:', PCA_6.explained_variance_ratio_)
print('PCA cumulative explained variance:', PCA_6.explained_variance_ratio_.cumsum())
print('LDA discriminatory-axis ratio:', LDA_1.explained_variance_ratio_)

component_stats = pd.DataFrame(X_PCA_6, columns=[f'PC{i+1}' for i in range(6)]).describe().T
print('PCA score statistics:')
display(component_stats)
print('LDA score statistics:')
display(pd.DataFrame(X_LDA_1, columns=['LD1']).describe().T)

In [ ]:
loadings = pd.DataFrame(PCA_6.components_.T,
                        index=mixed_features,
                        columns=[f'PC{i+1}' for i in range(6)])
print('Top 8 features (by absolute component weight):')
for component in loadings.columns:
    display(loadings[component].reindex(loadings[component].abs().sort_values(ascending=False).head(8).index).to_frame())

plt.figure(figsize=(9, 7))
sns.heatmap(loadings.loc[loadings.abs().max(axis=1).nlargest(20).index],
            cmap='coolwarm', center=0)
plt.title('Top PCA component weights (20 transformed features)')
plt.tight_layout()
plt.show()

In [ ]:
pca_plot = pd.DataFrame(X_PCA_6, columns=[f'PC{i+1}' for i in range(6)])
pca_plot[TARGET] = y.astype(str).to_numpy()
sns.pairplot(pca_plot, vars=[f'PC{i+1}' for i in range(6)], hue=TARGET,
             corner=True, diag_kind='hist',
             plot_kws={'alpha': 0.25, 's': 7})
plt.show()

plt.figure(figsize=(6, 5))
sns.heatmap(pd.DataFrame(X_PCA_6).corr(), annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1)
plt.title('PCA component correlation (should be nearly zero off-diagonal)')
plt.tight_layout()
plt.show()

In [ ]:
lda_plot = pd.DataFrame({'LD1': X_LDA_1[:, 0], TARGET: y.astype(str).to_numpy()})
plt.figure(figsize=(9, 4))
sns.histplot(data=lda_plot, x='LD1', hue=TARGET, stat='density',
             common_norm=False, bins=45, element='step', alpha=.35)
plt.title('LDA: distribution of the two Revenue classes')
plt.tight_layout()
plt.show()

**Interpretation:** PCA is unsupervised: the largest-variance axes need not separate the labels. LDA uses `Revenue` and produces **only one** axis for this binary target; its `explained_variance_ratio_ = [1]` describes the fraction of **discriminant variance**, **not** retention of 100% of total input variance. The LDA plot is **in-sample exploratory visualization**, not a generalization result.

### 2.2.3 TruncatedSVD on binary-encoded input

In [ ]:
SVD_6 = TruncatedSVD(n_components=6, random_state=RANDOM_STATE)
X_SVD_6 = SVD_6.fit_transform(X_binary)
print('SVD reduced shape:', X_SVD_6.shape)
svd_variance = pd.DataFrame({
    'Component': [f'SVD{i+1}' for i in range(6)],
    'Variance': SVD_6.explained_variance_,
    'Explained ratio': SVD_6.explained_variance_ratio_,
    'Cumulative ratio': np.cumsum(SVD_6.explained_variance_ratio_),
})
display(svd_variance)

In [ ]:
svd_plot = pd.DataFrame(X_SVD_6, columns=[f'SVD{i+1}' for i in range(6)])
svd_plot[TARGET] = y.astype(str).to_numpy()
sns.pairplot(svd_plot, vars=[f'SVD{i+1}' for i in range(6)], hue=TARGET,
             corner=True, diag_kind='hist',
             plot_kws={'alpha': 0.25, 's': 7})
plt.show()

**Interpretation:** TruncatedSVD can operate without centering and efficiently handles sparse one-hot input. Band-like groups in the plots reflect repeated/related binary patterns; color overlap indicates these groups do **not** map cleanly onto `Revenue`. No clear class separation is claimed from the unsupervised plots. The first SVD score need not have zero mean.

### 2.2.4 Information retained and number of components

In [ ]:
# PCA full spectrum. Results are for the exploratory full dataset.
pca_full = PCA(svd_solver='full').fit(X_mixed)
pca_cum = np.cumsum(pca_full.explained_variance_ratio_)

# Fit up to min(n_samples, n_features)-1 components for TruncatedSVD.
# For large matrices, use fewer components or randomized algorithm with a seed.
svd_full = TruncatedSVD(n_components=min(X_binary.shape)-1,
                        random_state=RANDOM_STATE, algorithm='randomized', n_iter=10)
svd_full.fit(X_binary)
svd_cum = np.cumsum(svd_full.explained_variance_ratio_)

thresholds = [0.80, 0.90, 0.95]
def dims_to_reach(cumulative, threshold):
    index = np.searchsorted(cumulative, threshold)
    return int(index + 1) if index < len(cumulative) else None

summary = pd.DataFrame({
    'Target variance': [f'{x:.0%}' for x in thresholds],
    'PCA components': [dims_to_reach(pca_cum, t) for t in thresholds],
    'SVD components': [dims_to_reach(svd_cum, t) for t in thresholds],
})
display(summary)

print('Input features: PCA/mixed =', X_mixed.shape[1],
      '; SVD/binary =', X_binary.shape[1])
print('Six-component variance: PCA =', round(pca_cum[5], 4),
      '; SVD =', round(svd_cum[5], 4))

plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(pca_cum)+1), pca_cum, label='PCA on Mixed')
plt.plot(np.arange(1, len(svd_cum)+1), svd_cum, label='TruncatedSVD on Binary')
for level in thresholds:
    plt.axhline(level, linestyle='--', alpha=.25)
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance ratio')
plt.ylim(0, 1.02)
plt.legend()
plt.title('Variance captured by each input representation')
plt.tight_layout()
plt.show()

**Important correction:** Recomputing from the current cleaned CSV and the current pipelines gives **PCA: 12/20/28 components** and **SVD: 22/33/42 components** for **80%/90%/95%** respectively. The old saved notebook output stating PCA 21/32/41 was inconsistent with this clean rerun (likely stale notebook state). Always use the freshly executed `summary` table. PCA (centered) and TruncatedSVD (uncentered) are applied to **different encodings**, so variance-ratio curves are not controlled comparisons of algorithms. Use a held-out classification metric to assess predictive value.

### 2.2.5 Class-label relationship and final method comparison

In [ ]:
# Exploratory, IN-SAMPLE univariate rank discrimination only;
# values >0.5 do NOT constitute validation accuracy.
rows = []
for name, scores in [('PCA', X_PCA_6), ('SVD', X_SVD_6), ('LDA', X_LDA_1)]:
    for j in range(scores.shape[1]):
        auc = roc_auc_score(y, scores[:, j])
        rows.append({'Method': name, 'Component': j + 1,
                     'In-sample univariate AUC (orientation-adjusted)': max(auc, 1-auc)})
display(pd.DataFrame(rows))

#### Conclusions for section 2.2

- **PCA (Mixed):** the first six components retain about **64.64%** of input variance. Their scatterplots show substantial class overlap and some skewness/outliers. The axes are linearly uncorrelated on the fitted input.
- **LDA (Mixed):** uses the class labels and returns just **one** discriminant direction (two Revenue classes). Analyze separation with the LD1 histogram; do not compare its `[1.]` discriminant ratio to PCA variance retention.
- **TruncatedSVD (Binary):** the first six components retain about **37.65%** of the variance in the transformed binary representation. Band-like or grouped patterns appear, but positive/negative Revenue labels strongly overlap.
- **At 95% explained variance (clean rerun):** **PCA: 28/75** and **TruncatedSVD: 42/105** dimensions, corresponding to **62.7% vs 60.0% feature-count reduction**, respectively, on **different input representations**. Earlier saved PCA counts of 41 at 95% should not be used without reproducing them.
- **Limitations:** PCA/SVD do not optimize class separation; apparent clusters do not imply Revenue classes. A high variance-retention ratio is not necessarily a high predictive score. Use train-only fitted pipelines in §2.4.

The requirement to visualize pairs of ~4–6 components is met by PCA and SVD; **LDA is necessarily 1D** in this binary problem and is displayed as a class-conditioned histogram instead.

---
## 2.3 Clustering — starter cells (complete the analysis here)

Use **at least one** of K-Means, GMM, DBSCAN; cluster **without `Revenue`**. Report quantitative input-space quality (e.g., silhouette) and how clusters relate to Revenue (e.g., contingency table, ARI/NMI as external descriptive checks); visualize predicted clusters **and** true classes with distinct color/markers. Do **not** cluster on LDA derived from labels as a supposedly unsupervised approach.

In [ ]:
# Example starting point (exploratory full-data fit; Revenue is excluded from features):
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import silhouette_samples

# PCA_6 was fitted without labels; appropriate for exploratory clustering.
cluster_features = X_PCA_6
kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(cluster_features)

print('Cluster sizes:', np.bincount(cluster_labels))
print('Cluster by Revenue:')
display(pd.crosstab(cluster_labels, y, normalize='index'))
print('ARI vs Revenue:', adjusted_rand_score(y, cluster_labels))
print('NMI vs Revenue:', normalized_mutual_info_score(y, cluster_labels))
# For runtime efficiency evaluate silhouette on a sample:
print('Silhouette (sample):', silhouette_score(cluster_features, cluster_labels,
                                              sample_size=2500, random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(cluster_features[:, 0], cluster_features[:, 1],
                c=cluster_labels, cmap='tab10', s=5, alpha=.35)
axes[0].set_title('K-Means clusters (PCA1–PCA2)')
axes[1].scatter(cluster_features[:, 0], cluster_features[:, 1],
                c=y.astype(int), cmap='coolwarm', s=5, alpha=.35)
axes[1].set_title('True Revenue labels (PCA1–PCA2)')
for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()

**To finish §2.3:** try several cluster counts and justify `k` using silhouette and cluster sizes; compare clustering on Mixed+PCA and Binary+SVD if useful; interpret which input attributes characterize clusters. Clustering labels are not expected to equal Revenue categories.

---
## 2.4 Classification — leakage-safe starter cells (complete the experiments here)

The brief requires at least **three classifiers** across the specified groups and train:validation splits **4:1, 7:3, 6:4**. Compare original and reduced feature sets, report performance and possible overfitting, tune regularization where applicable, and visualize predictions against actual labels. Use `stratify=y` because Revenue is imbalanced.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, roc_auc_score)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

models = {
    'BernoulliNB / binary': Pipeline([
        ('encoding', get_full_binary_pipeline()),
        ('clf', BernoulliNB(binarize=None))
    ]),
    'LogisticRegression / mixed': Pipeline([
        ('encoding', get_mixed_pipeline()),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
    ]),
    'SVM-RBF / mixed+PCA': Pipeline([
        ('encoding', get_mixed_pipeline()),
        ('reduction', PCA(n_components=20, random_state=RANDOM_STATE)),
        ('clf', SVC(kernel='rbf', class_weight='balanced'))
    ]),
    'LogisticRegression / binary+SVD': Pipeline([
        ('encoding', get_full_binary_pipeline()),
        ('reduction', TruncatedSVD(n_components=20, random_state=RANDOM_STATE)),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
    ])
}

# Run on demand; all preprocessing and reduction are fitted on TRAIN ONLY.
# For the required comparison, repeat each model for 0.2 / 0.3 / 0.4 test_size,
# add no-reduction baselines, and report F1, recall, average precision, overfit gaps.
# for name, model in models.items():
#     model.fit(X_train, y_train)
#     pred = model.predict(X_val)
#     print(name, '\n', classification_report(y_val, pred, zero_division=0))

**Model compatibility:** BernoulliNB works naturally on binary features, **not** directly on SVD continuous scores. GaussianNB requires dense data (e.g., Mixed+PCA or Binary+SVD). CategoricalNB uses integer nonnegative categories from Full Discrete and is not suitable for raw SVD scores. LDA as a supervised reducer must always fit within the training pipeline. Do not treat the global exploratory `X_PCA_6`, `X_LDA_1`, `X_SVD_6` as held-out model inputs.